In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)

n = 10000

# Generate base data
data = pd.DataFrame({
    "transaction_id": [f"TXN{1000000+i}" for i in range(n)],
    "record_number": np.arange(1, n+1),
    "origin_node_type": np.random.choice(["ADM", "EXT", "INT"], n),
    "origin_host_name": np.random.choice(["CMA", "CMB", "CMC"], n),
    "origin_operator_id": np.random.choice([1001, 1002, 1003], n),
    "host_name": np.random.choice(["HAAIR1", "HAAIR2", "HAAIR3", "HAAIR4"], n),
    "local_sequence_number": np.random.randint(10000000, 99999999, n),
    "current_service_class": np.random.choice(["100", "200", "300", "400", "500"], n),
    "voucher_based_refill": np.random.choice(["true", "false"], n),
    "transaction_type": np.random.choice(["0000000001", "0000000002", "0000000003"], n),
    "transaction_code": np.random.choice(["VREFL", "TOPUP", "DATA"], n),
    "transaction_currency": "DJF",
    "refill_type": np.random.choice(["0", "1"], n),
    "segmentation_id": np.random.choice(["100", "200", "300", "400", "500"], n),
    "voucher_group_id": np.random.choice(["v1", "v2", "v3"], n),
    "account_currency": "DJF",
    "voucher_agent": np.random.choice(["Evatis", "AgentA", "AgentB"], n),
})

# Generate timestamps
start_date = datetime(2024, 1, 1)
data["origin_timestamp"] = [
    start_date + timedelta(minutes=random.randint(0, 525600))
    for _ in range(n)
]
data["transaction_timestamp"] = data["origin_timestamp"]

# Subscriber numbers
data["subscriber_number"] = np.random.randint(77000000, 77999999, n).astype(str)
data["account_number"] = data["subscriber_number"]

# Voucher serial numbers
data["voucher_serial_number"] = np.random.randint(1000000000000, 9999999999999, n).astype(str)

# Elastic price logic (higher price → lower probability)
price_options = [100, 250, 500, 1000, 2000, 5000]
price_probs = [0.30, 0.25, 0.20, 0.15, 0.07, 0.03]  # Higher price less frequent

data["transaction_amount"] = np.random.choice(price_options, size=n, p=price_probs)

# Source file path
data["source_file_path"] = "/opt/airflow/data/raw/json/airfile/AIR2/2025/01/01/refillRecordV2.json"

# Processed timestamp
data["processed_at"] = datetime.now().isoformat()

# Save CSV
data.to_csv("price_elasticity.csv", index=False)

print("price_elasticity.csv generated successfully!")


price_elasticity.csv generated successfully!


In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv("price_elasticity.csv")

In [2]:
df.head(2)

,transaction_id,record_number,origin_node_type,origin_host_name,origin_operator_id,host_name,local_sequence_number,current_service_class,voucher_based_refill,transaction_type,...,account_currency,voucher_agent,origin_timestamp,transaction_timestamp,subscriber_number,account_number,voucher_serial_number,transaction_amount,source_file_path,processed_at
0,TXN1000000,1,INT,CMC,1002,HAAIR3,92845343,100,True,2,...,DJF,AgentB,2024-12-16 20:31:00,2024-12-16 20:31:00,77483983,77483983,6724438513651,250,/opt/airflow/data/raw/json/airfile/AIR2/2025/0...,2026-02-13T18:13:30.317575
1,TXN1000001,2,ADM,CMB,1002,HAAIR3,30069668,200,True,2,...,DJF,AgentA,2024-01-25 14:14:00,2024-01-25 14:14:00,77840973,77840973,6392132798171,500,/opt/airflow/data/raw/json/airfile/AIR2/2025/0...,2026-02-13T18:13:30.317575


In [6]:
df["refill_type"].unique()

array([0, 1])

In [7]:
df['transaction_amount'].value_counts()

transaction_amount
100     2987
250     2517
500     2000
1000    1486
2000     716
5000     294
Name: count, dtype: int64

In [8]:
df.groupby('voucher_agent')['transaction_amount'].sum()

voucher_agent
AgentA    2166150
AgentB    2059550
Evatis    2090250
Name: transaction_amount, dtype: int64

In [9]:
df['transaction_timestamp'] = pd.to_datetime(df['transaction_timestamp'])
df.set_index('transaction_timestamp', inplace=True)
df.resample('M')['transaction_amount'].count()

/tmp/ipykernel_11841/3553267238.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.resample('M')['transaction_amount'].count()


transaction_timestamp
2024-01-31    829
2024-02-29    797
2024-03-31    874
2024-04-30    860
2024-05-31    898
2024-06-30    796
2024-07-31    889
2024-08-31    838
2024-09-30    803
2024-10-31    782
2024-11-30    828
2024-12-31    806
Freq: ME, Name: transaction_amount, dtype: int64

In [10]:
df.groupby('subscriber_number')['transaction_amount'].agg(['count','sum','mean'])

,count,sum,mean
subscriber_number,,,
77000055,1,250,250.0
77000095,1,100,100.0
77000255,1,100,100.0
77000284,1,500,500.0
77000848,1,2000,2000.0
...,...,...,...
77999573,1,100,100.0
77999606,1,100,100.0
77999707,1,2000,2000.0


In [28]:
import pandas as pd

# Sample telecom data
df = pd.DataFrame([
    {"plan_id": 101, "price": 30, "subscriptions": 100},
    {"plan_id": 101, "price": 25, "subscriptions": 130},
    {"plan_id": 102, "price": 50, "subscriptions": 80},
    {"plan_id": 102, "price": 45, "subscriptions": 95}
])

# Function to calculate price elasticity
def price_elasticity(df, plan_id):
    plan_data = df[df['plan_id'] == plan_id].sort_values('price')
    print(f"plan_data {plan_data}")
    price_change = plan_data['price'].pct_change().iloc[0]
    print("         ")
    print(f"price_change {price_change}")
    demand_change = plan_data['subscriptions'].pct_change().iloc[1]
    print("         ")
    print(f"demand_change {demand_change}")
    elasticity = demand_change / price_change
    return elasticity

# Example: Elasticity for plan 101
e_101 = price_elasticity(df, 101)
print(f"Price Elasticity for Plan 101: {e_101:.2f}")


plan_data    plan_id  price  subscriptions
1      101     25            130
0      101     30            100
         
price_change 0.19999999999999996
         
demand_change -0.23076923076923073
Price Elasticity for Plan 101: -1.15


In [21]:
5/30

0.16666666666666666

In [26]:
import pandas as pd

data = {'A': [1, 2, 3], 'B': [4, 5, 6]}
df = pd.DataFrame(data)
print(df)
print("      ")
second_row = df.iloc[1]
print(second_row)

   A  B
0  1  4
1  2  5
2  3  6
      
A    2
B    5
Name: 1, dtype: int64
